<a href="https://colab.research.google.com/github/josephgonz12/March-Mania/blob/main/March_Mania.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

!pip install -q kaggle

!kaggle competitions download -c march-machine-learning-mania-2026

!unzip -q march-machine-learning-mania-2026.zip -d march-mania-data

100% 34.8M/34.8M [00:03<00:00, 10.7MB/s]



In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb

df_men = pd.read_csv('march-mania-data/MRegularSeasonCompactResults.csv')
df_women = pd.read_csv('march-mania-data/WRegularSeasonCompactResults.csv')

# Combine them into one master results dataframe
df_results = pd.concat([df_men, df_women], ignore_index=True)

# Load the submission template
df_sample_sub = pd.read_csv('march-mania-data/SampleSubmissionStage2.csv')

print(f"Total historical games loaded: {len(df_results)}")


#UF 1196

Total historical games loaded: 341084


In [ ]:
# Calculate basic Win/Loss counts
win_counts = df_results.groupby(['Season', 'WTeamID']).size().reset_index(name='Wins')
loss_counts = df_results.groupby(['Season', 'LTeamID']).size().reset_index(name='Losses')

win_counts.rename(columns={'WTeamID': 'TeamID'}, inplace=True)
loss_counts.rename(columns={'LTeamID': 'TeamID'}, inplace=True)

# Merge into a master team_stats dataframe
team_stats = pd.merge(win_counts, loss_counts, on=['Season', 'TeamID'], how='outer').fillna(0)
team_stats['Games'] = team_stats['Wins'] + team_stats['Losses']
team_stats['WinRate'] = team_stats['Wins'] / team_stats['Games']

# Keep only what we need from the compact era
team_stats = team_stats[['Season', 'TeamID', 'WinRate']]
print("Base Team Stats calculated successfully!")

Base Team Stats calculated successfully!


In [ ]:
df_men_det = pd.read_csv('march-mania-data/MRegularSeasonDetailedResults.csv')
df_women_det = pd.read_csv('march-mania-data/WRegularSeasonDetailedResults.csv')
df_det_results = pd.concat([df_men_det, df_women_det], ignore_index=True)

# Pre-calculate Rebounds and Possessions
df_det_results['WTR'] = df_det_results['WOR'] + df_det_results['WDR']
df_det_results['LTR'] = df_det_results['LOR'] + df_det_results['LDR']
df_det_results['WPoss'] = df_det_results['WFGA'] - df_det_results['WOR'] + df_det_results['WTO'] + (0.475 * df_det_results['WFTA'])
df_det_results['LPoss'] = df_det_results['LFGA'] - df_det_results['LOR'] + df_det_results['LTO'] + (0.475 * df_det_results['LFTA'])
df_det_results['GamePossessions'] = (df_det_results['WPoss'] + df_det_results['LPoss']) / 2

# Stack Winners and Losers (NOW PULLING 3-POINTERS AND STEALS!)
winners_det = df_det_results[['Season', 'WTeamID', 'WScore', 'LScore', 'GamePossessions', 'WFTM', 'WFTA', 'WTR', 'LTR', 'WTO', 'LTO', 'WFGA', 'WFGA3', 'WFGM3', 'WStl']].copy()
losers_det = df_det_results[['Season', 'LTeamID', 'LScore', 'WScore', 'GamePossessions', 'LFTM', 'LFTA', 'LTR', 'WTR', 'LTO', 'WTO', 'LFGA', 'LFGA3', 'LFGM3', 'LStl']].copy()

cols = ['Season', 'TeamID', 'PtsScored', 'PtsAllowed', 'Possessions', 'FTM', 'FTA', 'Reb', 'OppReb', 'TO', 'OppTO', 'FGA', 'FGA3', 'FGM3', 'Stl']
winners_det.columns = cols
losers_det.columns = cols

det_logs = pd.concat([winners_det, losers_det], ignore_index=True)

# Aggregate
det_stats = det_logs.groupby(['Season', 'TeamID']).agg(
    GamesPlayed=('PtsScored', 'count'), PtsScored=('PtsScored', 'sum'), PtsAllowed=('PtsAllowed', 'sum'),
    Possessions=('Possessions', 'sum'), FTM=('FTM', 'sum'), FTA=('FTA', 'sum'),
    Reb=('Reb', 'sum'), OppReb=('OppReb', 'sum'), TO=('TO', 'sum'), OppTO=('OppTO', 'sum'),
    FGA=('FGA', 'sum'), FGA3=('FGA3', 'sum'), FGM3=('FGM3', 'sum'), Stl=('Stl', 'sum')
).reset_index()

# Calculate Standard Advanced Metrics
det_stats['Pace'] = det_stats['Possessions'] / det_stats['GamesPlayed']
det_stats['OffRating'] = (det_stats['PtsScored'] / det_stats['Possessions']) * 100
det_stats['DefRating'] = (det_stats['PtsAllowed'] / det_stats['Possessions']) * 100
det_stats['FTPct'] = np.where(det_stats['FTA'] > 0, det_stats['FTM'] / det_stats['FTA'], 0)
det_stats['ReboundMargin'] = (det_stats['Reb'] - det_stats['OppReb']) / det_stats['GamesPlayed']
det_stats['TurnoverMargin'] = (det_stats['OppTO'] - det_stats['TO']) / det_stats['GamesPlayed']

# --- THE NEW UPSET INDICATORS ---
# What % of their shots are 3-pointers? (High variance style)
det_stats['ThreePtRate'] = np.where(det_stats['FGA'] > 0, det_stats['FGA3'] / det_stats['FGA'], 0)
# How well do they shoot them?
det_stats['ThreePtPct'] = np.where(det_stats['FGA3'] > 0, det_stats['FGM3'] / det_stats['FGA3'], 0)
# How often do they steal the ball per possession? (Disruption)
det_stats['StealRate'] = det_stats['Stl'] / det_stats['Possessions']
# --------------------------------

# Merge and fill NAs
custom_features = det_stats[['Season', 'TeamID', 'Pace', 'OffRating', 'DefRating', 'FTPct', 'ReboundMargin', 'TurnoverMargin', 'ThreePtRate', 'ThreePtPct', 'StealRate']]
team_stats = pd.merge(team_stats, custom_features, on=['Season', 'TeamID'], how='left')

team_stats.fillna({'Pace': team_stats['Pace'].mean(), 'OffRating': 100, 'DefRating': 100, 'FTPct': 0.70, 'ReboundMargin': 0, 'TurnoverMargin': 0, 'ThreePtRate': 0.33, 'ThreePtPct': 0.33, 'StealRate': 0.08}, inplace=True)

print("Advanced Metrics and Upset Indicators fully integrated!")

Advanced Metrics and Upset Indicators fully integrated!


In [ ]:
df_men_seeds = pd.read_csv('march-mania-data/MNCAATourneySeeds.csv')
df_women_seeds = pd.read_csv('march-mania-data/WNCAATourneySeeds.csv')
df_seeds = pd.concat([df_men_seeds, df_women_seeds], ignore_index=True)

# Added the 'r' before the string to fix the invalid escape sequence warning!
df_seeds['SeedNum'] = df_seeds['Seed'].str.extract(r'(\d+)').astype(int)

# Keep only what we need
df_seeds = df_seeds[['Season', 'TeamID', 'SeedNum']]
print("Tournament Seeds processed!")

Tournament Seeds processed!


In [ ]:
# Setup Winners (Target 1) and Losers (Target 0)
df_men_tourney = pd.read_csv('march-mania-data/MNCAATourneyCompactResults.csv')
df_women_tourney = pd.read_csv('march-mania-data/WNCAATourneyCompactResults.csv')
df_tourney_results = pd.concat([df_men_tourney, df_women_tourney], ignore_index=True)

# Setup Winners and Losers
df_win = df_tourney_results[['Season', 'WTeamID', 'LTeamID']].copy()
df_win.rename(columns={'WTeamID': 'TeamA', 'LTeamID': 'TeamB'}, inplace=True)
df_win['Target'] = 1

df_loss = df_tourney_results[['Season', 'LTeamID', 'WTeamID']].copy()
df_loss.rename(columns={'LTeamID': 'TeamA', 'WTeamID': 'TeamB'}, inplace=True)
df_loss['Target'] = 0

df_train = pd.concat([df_win, df_loss], ignore_index=True)

# Merge Advanced Stats for Team A (NOW INCLUDING UPSET INDICATORS)
df_train = pd.merge(df_train, team_stats, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'], how='left').drop(columns=['TeamID'])
df_train.rename(columns={
    'WinRate': 'WinRateA', 'Pace': 'PaceA', 'OffRating': 'OffRatingA',
    'DefRating': 'DefRatingA', 'FTPct': 'FTPctA', 'ReboundMargin': 'ReboundMarginA',
    'TurnoverMargin': 'TurnoverMarginA',
    'ThreePtRate': 'ThreePtRateA', 'ThreePtPct': 'ThreePtPctA', 'StealRate': 'StealRateA'
}, inplace=True)

# Merge Advanced Stats for Team B
df_train = pd.merge(df_train, team_stats, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'], how='left').drop(columns=['TeamID'])
df_train.rename(columns={
    'WinRate': 'WinRateB', 'Pace': 'PaceB', 'OffRating': 'OffRatingB',
    'DefRating': 'DefRatingB', 'FTPct': 'FTPctB', 'ReboundMargin': 'ReboundMarginB',
    'TurnoverMargin': 'TurnoverMarginB',
    'ThreePtRate': 'ThreePtRateB', 'ThreePtPct': 'ThreePtPctB', 'StealRate': 'StealRateB'
}, inplace=True)

# MERGE SEEDS!
df_train = pd.merge(df_train, df_seeds, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'], how='left').drop(columns=['TeamID'])
df_train.rename(columns={'SeedNum': 'SeedA'}, inplace=True)

df_train = pd.merge(df_train, df_seeds, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'], how='left').drop(columns=['TeamID'])
df_train.rename(columns={'SeedNum': 'SeedB'}, inplace=True)

# Calculate all differences
df_train['WinRateDiff'] = df_train['WinRateA'] - df_train['WinRateB']
df_train['PaceDiff'] = df_train['PaceA'] - df_train['PaceB']
df_train['OffRtgDiff'] = df_train['OffRatingA'] - df_train['OffRatingB']
df_train['DefRtgDiff'] = df_train['DefRatingA'] - df_train['DefRatingB']
df_train['FTPctDiff'] = df_train['FTPctA'] - df_train['FTPctB']
df_train['RebMarginDiff'] = df_train['ReboundMarginA'] - df_train['ReboundMarginB']
df_train['TOMarginDiff'] = df_train['TurnoverMarginA'] - df_train['TurnoverMarginB']
df_train['SeedDiff'] = df_train['SeedA'] - df_train['SeedB']

# The Upset Indicators
df_train['ThreePtRateDiff'] = df_train['ThreePtRateA'] - df_train['ThreePtRateB']
df_train['ThreePtPctDiff'] = df_train['ThreePtPctA'] - df_train['ThreePtPctB']
df_train['StealRateDiff'] = df_train['StealRateA'] - df_train['StealRateB']

# Drop rows with NaN
df_train = df_train.dropna()

print(f"Tournament training dataset ready with {len(df_train)} matchup rows!")

Tournament training dataset ready with 8604 matchup rows!


In [ ]:
features = [
    'SeedDiff',
    'WinRateDiff',
    'PaceDiff',
    'OffRtgDiff',
    'DefRtgDiff',
    'FTPctDiff',
    'RebMarginDiff',
    'TOMarginDiff',
    'ThreePtRateDiff', # <--- Upset Indicator
    'ThreePtPctDiff',  # <--- Upset Indicator
    'StealRateDiff'    # <--- Upset Indicator
]

X = df_train[features]
y = df_train['Target']

model = xgb.XGBClassifier(
    n_estimators=350,
    learning_rate=0.08,
    max_depth=9,
    subsample=0.56,
    colsample_bytree=0.85,
    objective='binary:logistic',
    eval_metric='logloss'
)

print(f"Training XGBoost Model on {len(features)} features...")
model.fit(X, y)

# Let's see what the model thinks is most important!
importance = pd.DataFrame({'Feature': features, 'Importance': model.feature_importances_})
print("\nFeature Importance:")
print(importance.sort_values(by='Importance', ascending=False).to_string(index=False))

Training XGBoost Model on 11 features...

Feature Importance:
        Feature  Importance
       SeedDiff    0.210595
     OffRtgDiff    0.093795
     DefRtgDiff    0.081995
    WinRateDiff    0.078002
   TOMarginDiff    0.077385
       PaceDiff    0.077309
  RebMarginDiff    0.077216
 ThreePtPctDiff    0.076948
ThreePtRateDiff    0.075918
      FTPctDiff    0.075622
  StealRateDiff    0.075215


In [ ]:
# Parse the ID format
sub_split = df_sample_sub['ID'].str.split('_', expand=True)
df_sample_sub['Season'] = sub_split[0].astype(int)
df_sample_sub['TeamA'] = sub_split[1].astype(int)
df_sample_sub['TeamB'] = sub_split[2].astype(int)

# Merge stats for Team A
df_sub = pd.merge(df_sample_sub, team_stats, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'], how='left').drop(columns=['TeamID'])
df_sub.rename(columns={
    'WinRate': 'WinRateA', 'Pace': 'PaceA', 'OffRating': 'OffRatingA',
    'DefRating': 'DefRatingA', 'FTPct': 'FTPctA', 'ReboundMargin': 'ReboundMarginA',
    'TurnoverMargin': 'TurnoverMarginA',
    'ThreePtRate': 'ThreePtRateA', 'ThreePtPct': 'ThreePtPctA', 'StealRate': 'StealRateA'
}, inplace=True)

# Merge stats for Team B
df_sub = pd.merge(df_sub, team_stats, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'], how='left').drop(columns=['TeamID'])
df_sub.rename(columns={
    'WinRate': 'WinRateB', 'Pace': 'PaceB', 'OffRating': 'OffRatingB',
    'DefRating': 'DefRatingB', 'FTPct': 'FTPctB', 'ReboundMargin': 'ReboundMarginB',
    'TurnoverMargin': 'TurnoverMarginB',
    'ThreePtRate': 'ThreePtRateB', 'ThreePtPct': 'ThreePtPctB', 'StealRate': 'StealRateB'
}, inplace=True)

# MERGE SEEDS for 2026
df_sub = pd.merge(df_sub, df_seeds, left_on=['Season', 'TeamA'], right_on=['Season', 'TeamID'], how='left').drop(columns=['TeamID'])
df_sub.rename(columns={'SeedNum': 'SeedA'}, inplace=True)

df_sub = pd.merge(df_sub, df_seeds, left_on=['Season', 'TeamB'], right_on=['Season', 'TeamID'], how='left').drop(columns=['TeamID'])
df_sub.rename(columns={'SeedNum': 'SeedB'}, inplace=True)

# Calculate the differences
df_sub['WinRateDiff'] = df_sub['WinRateA'] - df_sub['WinRateB']
df_sub['PaceDiff'] = df_sub['PaceA'] - df_sub['PaceB']
df_sub['OffRtgDiff'] = df_sub['OffRatingA'] - df_sub['OffRatingB']
df_sub['DefRtgDiff'] = df_sub['DefRatingA'] - df_sub['DefRatingB']
df_sub['FTPctDiff'] = df_sub['FTPctA'] - df_sub['FTPctB']
df_sub['RebMarginDiff'] = df_sub['ReboundMarginA'] - df_sub['ReboundMarginB']
df_sub['TOMarginDiff'] = df_sub['TurnoverMarginA'] - df_sub['TurnoverMarginB']
df_sub['SeedDiff'] = df_sub['SeedA'] - df_sub['SeedB']

df_sub['ThreePtRateDiff'] = df_sub['ThreePtRateA'] - df_sub['ThreePtRateB']
df_sub['ThreePtPctDiff'] = df_sub['ThreePtPctA'] - df_sub['ThreePtPctB']
df_sub['StealRateDiff'] = df_sub['StealRateA'] - df_sub['StealRateB']

# Predict the Probabilities
print("Generating 2026 predictions with Upset Indicators...")
features = ['SeedDiff', 'WinRateDiff', 'PaceDiff', 'OffRtgDiff', 'DefRtgDiff', 'FTPctDiff', 'RebMarginDiff', 'TOMarginDiff', 'ThreePtRateDiff', 'ThreePtPctDiff', 'StealRateDiff']
df_sub['Pred'] = model.predict_proba(df_sub[features])[:, 1]

# --- THE GATOR OVERRIDE ---
# print("Applying the Florida Gators 100% win-rate override...")
# df_sub.loc[df_sub['TeamA'] == 1196, 'Pred'] = 1.0
# df_sub.loc[df_sub['TeamB'] == 1196, 'Pred'] = 0.0

# Format and save
final_submission = df_sub[['ID', 'Pred']]
final_submission.to_csv('my_final_mania_submission.csv', index=False)
print("Submission saved! Run the bracket simulator again.")

Generating 2026 predictions with Upset Indicators...
Submission saved! Run the bracket simulator again.


In [ ]:
def simulate_perfect_bracket(gender='M'):
    print(f"\n{'='*50}")
    print(f"🏀 SIMULATING FULL 2026 {gender}EN'S TOURNAMENT 🏀")
    print(f"{'='*50}\n")

    # Load your submission and the Kaggle structural files
    sub = pd.read_csv('my_final_mania_submission.csv')
    teams = pd.read_csv('march-mania-data/MTeams.csv')
    seeds = pd.read_csv(f'march-mania-data/{gender}NCAATourneySeeds.csv')
    slots = pd.read_csv(f'march-mania-data/{gender}NCAATourneySlots.csv')

    # Filter for the current 2026 season
    seeds = seeds[seeds['Season'] == 2026]
    slots = slots[slots['Season'] == 2026].copy()

    # --- THE FIX: SORT CHRONOLOGICALLY ---
    # First Four games don't start with 'R' (e.g., W16). We assign them Round 0.
    def get_round(slot):
        if slot.startswith('R'):
            return int(slot[1]) # 'R1W1' -> 1
        return 0 # First Four -> 0

    slots['RoundNum'] = slots['Slot'].apply(get_round)
    slots = slots.sort_values(by=['RoundNum', 'Slot'])
    # -------------------------------------

    # Dictionaries for lookups
    team_dict = dict(zip(teams['TeamID'], teams['TeamName']))
    current_seeds = dict(zip(seeds['Seed'], seeds['TeamID']))

    def get_winner(teamA_id, teamB_id):
        if teamA_id < teamB_id:
            matchup_id = f"2026_{teamA_id}_{teamB_id}"
            prob = sub.loc[sub['ID'] == matchup_id, 'Pred'].values[0]
            return teamA_id if prob >= 0.5 else teamB_id
        else:
            matchup_id = f"2026_{teamB_id}_{teamA_id}"
            prob = sub.loc[sub['ID'] == matchup_id, 'Pred'].values[0]
            return teamB_id if prob >= 0.5 else teamA_id

    current_round_header = ""

    # Now we can just do a standard top-to-bottom loop!
    for index, row in slots.iterrows():
        slot = row['Slot']
        strong = row['StrongSeed']
        weak = row['WeakSeed']

        # Because we sorted by Round, the dependencies are guaranteed to be met!
        if strong in current_seeds and weak in current_seeds:
            teamA = current_seeds[strong]
            teamB = current_seeds[weak]

            winner = get_winner(teamA, teamB)
            current_seeds[slot] = winner # Advance the winner

            # Determine round name for clean printing
            if slot.startswith('R1'): round_name = "ROUND 1 (Round of 64)"
            elif slot.startswith('R2'): round_name = "ROUND 2 (Round of 32)"
            elif slot.startswith('R3'): round_name = "SWEET 16"
            elif slot.startswith('R4'): round_name = "ELITE 8"
            elif slot.startswith('R5'): round_name = "FINAL FOUR"
            elif slot.startswith('R6'): round_name = "NATIONAL CHAMPIONSHIP"
            else: round_name = "FIRST FOUR (Play-In)"

            if round_name != current_round_header:
                print(f"\n--- {round_name} ---")
                current_round_header = round_name

            # Print the matchup
            print(f"{team_dict[teamA]}  vs  {team_dict[teamB]}")
            print(f"   ---> Winner: {team_dict[winner]}")
        else:
            print(f"Error: Could not find seeds for slot {slot}")

# Run the simulation!
simulate_perfect_bracket('M')


🏀 SIMULATING FULL 2026 MEN'S TOURNAMENT 🏀


--- FIRST FOUR (Play-In) ---
Lehigh  vs  Prairie View
   ---> Winner: Prairie View
Miami OH  vs  SMU
   ---> Winner: Miami OH
Howard  vs  UMBC
   ---> Winner: Howard
NC State  vs  Texas
   ---> Winner: Texas

--- ROUND 1 (Round of 64) ---
Duke  vs  Siena
   ---> Winner: Duke
Connecticut  vs  Furman
   ---> Winner: Connecticut
Michigan St  vs  N Dakota St
   ---> Winner: Michigan St
Kansas  vs  Cal Baptist
   ---> Winner: Kansas
St John's  vs  Northern Iowa
   ---> Winner: St John's
Louisville  vs  South Florida
   ---> Winner: Louisville
UCLA  vs  UCF
   ---> Winner: UCLA
Ohio St  vs  TCU
   ---> Winner: TCU
Florida  vs  Prairie View
   ---> Winner: Florida
Houston  vs  Idaho
   ---> Winner: Houston
Illinois  vs  Penn
   ---> Winner: Illinois
Nebraska  vs  Troy
   ---> Winner: Nebraska
Vanderbilt  vs  McNeese St
   ---> Winner: McNeese St
North Carolina  vs  VCU
   ---> Winner: North Carolina
St Mary's CA  vs  Texas A&M
   ---> Winner: St Ma